In [1]:
!pip install -q faiss-gpu-cu12 sentence-transformers transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest-asyncio sentencepiece sacremoses googletrans==4.0.0-rc1 httpcore==0.9.1

In [6]:
import json
import re
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, MarianMTModel, MarianTokenizer
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from kaggle_secrets import UserSecretsClient

# Setup Ngrok Token from Kaggle Secrets
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(ngrok_token)

In [9]:
# ── Translation Model (Hinglish → English) ───────────────
from googletrans import Translator
import asyncio

# Initialize the translator
translator = Translator()

def translate_to_english(text: str) -> str:
    """
    Converts Hinglish (Roman script) → English using Google Translate.
    Automatically detects language so works for:
    - Roman Hinglish  (mera account hack ho gaya)
    - Devanagari Hindi (मेरा अकाउंट हैक हो गया)
    - Mixed Hindi+English sentences
    - Pure English (returned as-is)
    """
    try:
        result = translator.translate(text, src='hi', dest='en')
        return result.text
    except Exception as e:
        # If translation fails for any reason, return original text
        print(f"  [Translation warning: {e}]")
        return text

print("✅ Translation model ready")

# ── Quick test ────────────────────────────────────────────
test_sentences = [
    "Sir mera account hack ho gaya aur Rs 45,000 nikal gaye",
    "Kisi ne mujhe UPI ke through paise chura liye",
    "Mere ghar mein ghuskar laptop chura liya gaya",
    "Neighbour ne mujhpe haath uthaya aur dhamki di",
    "mere dost nee dusre dost koo chaku se marr diya",
    "char logon ne milke ek ladki ka balaatkaar kiya"
]

print()
for t in test_sentences:
    result = translate_to_english(t)
    print(f"IN : {t}")
    print(f"OUT: {result}")
    print()

✅ Translation model ready

IN : Sir mera account hack ho gaya aur Rs 45,000 nikal gaye
OUT: Sir my account got hacked and Rs 45,000 was withdrawn

IN : Kisi ne mujhe UPI ke through paise chura liye
OUT: Someone stole money from me through UPI

IN : Mere ghar mein ghuskar laptop chura liya gaya
OUT: My laptop was broken into and stolen

IN : Neighbour ne mujhpe haath uthaya aur dhamki di
OUT: The overseer raised his hand on me and threatened me

IN : mere dost nee dusre dost koo chaku se marr diya
OUT: my friend killed another friend with a knife

IN : char logon ne milke ek ladki ka balaatkaar kiya
OUT: Four people together raped a girl



In [13]:
# 1. Master Lists
crime_categories = [
    "Homicide", "Attempted Murder", "Aggravated Assault", "Simple Assault", 
    "Kidnapping", "Sexual Assault", "Domestic Violence", "Burglary", 
    "Larceny/Theft", "Motor Vehicle Theft", "Arson", "Vandalism/Property Damage", 
    "Trespassing", "Fraud/Deception", "Cybercrime/Hacking", "Identity Theft", 
    "Extortion/Blackmail", "Embezzlement", "Drug Trafficking", "Drug Possession", 
    "Weapons Offenses", "Disorderly Conduct", "Traffic/DUI", "Hit and Run", 
    "Stalking", "Harassment"
]

severity_levels = ["Low", "Medium", "High", "Critical"]

# 2. Knowledge Base
knowledge_base = [
    {
        "complaint": "Someone smashed my car window and took the radio.", 
        "categories": ["Larceny/Theft", "Vandalism/Property Damage"],
        "severity": "Medium"
    },
    {
        "complaint": "A man aggressively approached me with a knife, grabbed my purse, and ran.", 
        "categories": ["Robbery", "Weapons Offenses"],
        "severity": "High"
    },
    {
        "complaint": "A drunk driver T-boned my sedan and sped off, leaving my passenger severely injured.", 
        "categories": ["Traffic/DUI", "Hit and Run", "Aggravated Assault"],
        "severity": "Critical"
    },
    {
        "complaint": "Teenagers are skateboarding on my property and knocked over my trash cans.", 
        "categories": ["Trespassing", "Disorderly Conduct"],
        "severity": "Low"
    },
    {
        "complaint": "I received an email demanding ₹10 lakhs in Bitcoin or my private files will be leaked.", 
        "categories": ["Extortion/Blackmail", "Cybercrime/Hacking"],
        "severity": "High"
    },
    {
        "complaint": "Someone broke into my electronics shop overnight and stole inventory worth ₹2 crores.",
        "categories": ["Burglary", "Larceny/Theft"],
        "severity": "High"
    },
    {
        "complaint": "My bank account was drained of ₹5 lakhs after someone cloned my debit card and bypassed the OTP.",
        "categories": ["Cybercrime/Hacking", "Fraud/Deception", "Identity Theft"],
        "severity": "High"
    },
    {
        "complaint": "My spouse beat me severely during an argument and locked me in the bedroom for two days.",
        "categories": ["Domestic Violence", "Aggravated Assault", "Kidnapping"],
        "severity": "Critical"
    },
    {
        "complaint": "Police pulled over a suspicious truck and found 50kg of illegal narcotics hidden under the seats along with an unlicensed pistol.",
        "categories": ["Drug Trafficking", "Weapons Offenses"],
        "severity": "Critical"
    },
    {
        "complaint": "My ex-boyfriend follows me from the metro station to my office every day and leaves hundreds of missed calls.",
        "categories": ["Stalking", "Harassment"],
        "severity": "Medium"
    },
    {
        "complaint": "An audit revealed that the lead accountant siphoned off ₹1.5 crores from company funds over the last three years.",
        "categories": ["Embezzlement", "Fraud/Deception"],
        "severity": "High"
    },
    {
        "complaint": "Someone spray-painted obscenities on the community compound wall late last night.",
        "categories": ["Vandalism/Property Damage"],
        "severity": "Low"
    },
    {
        "complaint": "A speeding commercial truck crashed into my parked motorcycle and drove away without stopping.",
        "categories": ["Hit and Run", "Vandalism/Property Damage"],
        "severity": "Medium"
    },
    {
        "complaint": "They abducted my son on his way to school and are demanding a ₹5 crore ransom for his safe return.",
        "categories": ["Kidnapping", "Extortion/Blackmail"],
        "severity": "Critical"
    },
    {
        "complaint": "During a street fight outside the pub, a man pulled out a gun and shot another person in the chest.",
        "categories": ["Attempted Murder", "Aggravated Assault", "Weapons Offenses"],
        "severity": "Critical"
    },
    {
        "complaint": "The factory owner intentionally set fire to the empty warehouse to falsely claim the ₹10 crore insurance policy.",
        "categories": ["Arson", "Fraud/Deception"],
        "severity": "High"
    },
    {
        "complaint": "I found two people smoking illicit drugs inside the abandoned construction site behind my apartment.",
        "categories": ["Drug Possession", "Trespassing"],
        "severity": "Medium"
    },
    {
        "complaint": "A hacker took over my Instagram account and is posting morphed, inappropriate photos to ruin my reputation.",
        "categories": ["Cybercrime/Hacking", "Harassment", "Identity Theft"],
        "severity": "High"
    },
    {
        "complaint": "A dead body was discovered in the river this morning with multiple deep stab wounds.",
        "categories": ["Homicide"],
        "severity": "Critical"
    },
    {
        "complaint": "A customer walked out of the retail store with a smartwatch hidden inside his jacket pocket.",
        "categories": ["Larceny/Theft"],
        "severity": "Low"
    },
    {
        "complaint": "A car is swerving wildly across lanes on the highway, ignoring traffic signals, and almost hit the divider.",
        "categories": ["Traffic/DUI", "Disorderly Conduct"],
        "severity": "Medium"
    },
    {
        "complaint": "I parked my scooter outside the grocery store for ten minutes, and when I came back, it was completely gone.",
        "categories": ["Motor Vehicle Theft"],
        "severity": "Medium"
    },
    {
        "complaint": "My neighbor started screaming at me, threw a rock through my living room window, and threatened to beat me up.",
        "categories": ["Simple Assault", "Vandalism/Property Damage", "Harassment"],
        "severity": "Medium"
    },
    {
        "complaint": "A woman was pulled into an alleyway against her will and sexually assaulted by an unidentified suspect.",
        "categories": ["Sexual Assault", "Kidnapping", "Aggravated Assault"],
        "severity": "Critical"
    },
    {
        "complaint": "A group of people are standing outside the storefront shouting obscenities at customers and refusing to leave.",
        "categories": ["Disorderly Conduct", "Trespassing"],
        "severity": "Low"
    }
]

In [14]:
# 3. Rebuild FAISS Index (Fixes the IndexError)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [item["complaint"] for item in knowledge_base]
embeddings = embedder.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# 4. Load Open Model
model_id = "HuggingFaceH4/zephyr-7b-beta"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. Classification Logic
def classify_complaint(new_complaint, top_k=2):
    new_embed = embedder.encode([new_complaint])
    distances, indices = index.search(np.array(new_embed), top_k)
    
    few_shot_context = ""
    for idx in indices[0]:
        # Fix: Ignore out-of-bounds or -1 indices
        if idx == -1 or idx >= len(knowledge_base):
            continue
            
        match = knowledge_base[idx]
        example_json = json.dumps({"categories": match['categories'], "severity": match['severity']})
        few_shot_context += f"Complaint: {match['complaint']}\nOutput: {example_json}\n\n"

    system_prompt = f"""<|system|>
You are a strict law enforcement AI. Classify the complaint into one or more categories: {json.dumps(crime_categories)}.
Assign a severity: {json.dumps(severity_levels)}.
Output ONLY a valid JSON object. Do not include markdown formatting, explanations, or introductory text.</s>
<|user|>
Examples:
{few_shot_context}
New Complaint: {new_complaint}
Output:</s>
<|assistant|>"""

    inputs = tokenizer(system_prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, 
        max_new_tokens=60, 
        do_sample=True, 
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    try:
        json_match = re.search(r'\{.*?\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return {"error": "Could not parse JSON", "raw_output": response}
    except json.JSONDecodeError:
        return {"error": "Invalid JSON generated", "raw_output": response}

# 6. Test
test_text = "Someone broke into my house while I was sleeping and stole my laptop and ₹50,000 in cash."
print(classify_complaint(test_text))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model-00005-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00007-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

{'categories': ['Burglary', 'Larceny/Theft'], 'severity': 'Medium'}


In [17]:
# ── Full pipeline: translate → classify ──────────────────
def translate_and_classify(hinglish_text: str) -> dict:
    english_text = translate_to_english(hinglish_text)
    result = classify_complaint(english_text)
    return {
        "original_hinglish" : hinglish_text,
        "translated_english": english_text,
        "classification"    : result
    }

# ── Test the full pipeline ────────────────────────────────
tests = [
    "Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye",
    "Ek aadmi ne mujhe UPI pe fake collect request bheja aur Rs 22,000 chale gaye",
    "Mere ghar mein ghuskar laptop aur nakdi chura li gayi",
    "Neighbour ne mujhpe haath uthaya aur dhamki di",
]

print("=" * 55)
print("FULL PIPELINE TEST — Translation + Classification")
print("=" * 55)

for i, complaint in enumerate(tests, 1):
    print(f"\n── Test {i} ──────────────────────────────────")
    out = translate_and_classify(complaint)
    print(f"  ORIGINAL  : {out['original_hinglish']}")
    print(f"  TRANSLATED: {out['translated_english']}")
    print(f"  CATEGORIES: {out['classification'].get('categories', 'N/A')}")
    print(f"  SEVERITY  : {out['classification'].get('severity', 'N/A')}")

print("\n" + "=" * 55)
print("✅ Pipeline test complete")

FULL PIPELINE TEST — Translation + Classification

── Test 1 ──────────────────────────────────
  ORIGINAL  : Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye
  TRANSLATED: Sir my bank account got hacked and someone withdrew Rs 45,000
  CATEGORIES: ['Cybercrime/Hacking', 'Fraud/Deception', 'Identity Theft']
  SEVERITY  : High

── Test 2 ──────────────────────────────────
  ORIGINAL  : Ek aadmi ne mujhe UPI pe fake collect request bheja aur Rs 22,000 chale gaye
  TRANSLATED: A guy sent me a fake collect request on UPI and Rs 22,000 was gone
  CATEGORIES: ['Fraud/Deception', 'Cybercrime/Hacking']
  SEVERITY  : Medium

── Test 3 ──────────────────────────────────
  ORIGINAL  : Mere ghar mein ghuskar laptop aur nakdi chura li gayi
  TRANSLATED: My house was broken into and laptop and cash were stolen
  CATEGORIES: ['Burglary', 'Larceny/Theft']
  SEVERITY  : Medium

── Test 4 ──────────────────────────────────
  ORIGINAL  : Neighbour ne mujhpe haath uthaya aur dhamki di
 

In [ ]:
import asyncio
import uvicorn

app = FastAPI()

class ComplaintRequest(BaseModel):
    text: str

@app.post("/classify")
async def classify_api(request: ComplaintRequest):
    result = classify_complaint(request.text)
    return result

# 1. Start Tunnel
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 API LIVE AT: {public_url}/classify\n")

# 2. Manual Server Setup (Fixes the RuntimeError)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# 3. Apply nest_asyncio and run within the existing loop
nest_asyncio.apply()
await server.serve()


🚀 API LIVE AT: https://innermostly-paranormal-idell.ngrok-free.dev/classify



INFO:     Started server process [488]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2405:201:c008:1167:10e0:41ff:e66d:58f6:0 - "GET /classify HTTP/1.1" 405 Method Not Allowed
INFO:     2405:201:c008:1167:10e0:41ff:e66d:58f6:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2405:201:c008:1167:10e0:41ff:e66d:58f6:0 - "GET /classify HTTP/1.1" 405 Method Not Allowed
